# Controlled model comparison (issue #45)

Two distinct approved approaches, a regularized linear model (E01) and gradient-boosted trees (E02), are trained on the frozen `OCT-SPLIT-001` training partition and compared against the required training-mean constant and the training-median comparator.

**Sources this notebook builds on (all by teammates, read before writing):**
- Split and baselines: `docs/team/october/Gate 1/split_design_decision.md`, `notebooks/final-deliverables/October/Gate 2/` (Ella Zou, Liam Stapley)
- Preprocessing and experiment plan (E01/E02 definitions, rare-level threshold 132, categorical mask): `docs/team/october/Gate 1/preprocessing_plan.md`, `experiment_plan.md` (Mia Chavez, branch `42-preprocessing-and-experiment-plan`)
- Cross-check only: Jonathan Deng's `notebooks/jonathan-deng/october_modeling_outputs/` (branch `jonathan`). His numbers are compared in section 10; nothing here is copied from them.

**Rules kept:** fit only on train; tune only with the 5 recorded `cv_fold` folds (pipeline refit per fold); validation is a limited comparison check; test rows are dropped immediately after load and never summarised, fitted or scored; MAE is in original `loss` units; seed 42.

In [1]:
import hashlib, io, json, platform, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 42
RARE_MIN_FREQUENCY = 132  # 0.1% of train, from preprocessing_plan.md
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "allstate_claims_data.csv"
MANIFEST = ROOT / "notebooks" / "final-deliverables" / "October" / "Gate 1" / "partition_manifest.csv"
OUT = ROOT / "notebooks" / "junaid-pathan" / "october" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED_SOURCE_SHA = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
EXPECTED_MANIFEST_SHA = "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b"
WORKFLOW_RUN = "OCT-JP-MODEL-COMPARISON-001"

def sha256(path):
    # Frozen hashes were taken on LF files; normalise CRLF so a Windows checkout (autocrlf) compares equal.
    data = Path(path).read_bytes().replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()

assert sha256(DATA) == EXPECTED_SOURCE_SHA, "source file differs from the frozen version"
assert sha256(MANIFEST) == EXPECTED_MANIFEST_SHA, "manifest differs from OCT-SPLIT-001"
versions = {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
            "pandas": pd.__version__, "numpy": np.__version__}
print(versions)
if sklearn.__version__ != "1.6.1":
    print("NOTE: preprocessing_plan.md pins scikit-learn 1.6.1; this run used", sklearn.__version__)

{'python': '3.10.0', 'scikit-learn': '1.7.2', 'pandas': '2.3.3', 'numpy': '2.2.6'}
NOTE: preprocessing_plan.md pins scikit-learn 1.6.1; this run used 1.7.2


## 1. Load the frozen split
Merge the manifest on `id`, then drop the protected test rows immediately.

In [2]:
source = pd.read_csv(DATA)
manifest = pd.read_csv(MANIFEST)
df = source.merge(manifest, on="id", how="inner", validate="one_to_one")
df = df[df["partition"] != "test"].reset_index(drop=True)   # test rows are dropped here; their loss is never summarised, fitted or scored

train = df[df["partition"] == "train"].reset_index(drop=True)
validation = df[df["partition"] == "validation"].reset_index(drop=True)
CAT = [f"cat{i}" for i in range(1, 117)]
CONT = [f"cont{i}" for i in range(1, 15)]
FEATURES = CAT + CONT

assert (len(train), len(validation)) == (131_822, 28_248)
assert set(df["partition"]) == {"train", "validation"}
assert train["cv_fold"].isin(range(5)).all() and validation["cv_fold"].isna().all()
assert df[FEATURES].isna().sum().sum() == 0          # missing-value guard from the plan
assert "loss" not in FEATURES and "id" not in FEATURES
print(train["cv_fold"].value_counts().sort_index().to_dict())
pd.DataFrame({"partition": ["train", "validation"], "claims": [len(train), len(validation)]})

{0.0: 26365, 1.0: 26365, 2.0: 26364, 3.0: 26364, 4.0: 26364}


,partition,claims
0,train,131822
1,validation,28248


## 2. Shared evaluation design
One function scores every model, so baselines and candidates use the same claims and the same original-unit MAE. Cross-validation uses the manifest's `cv_fold` column and refits the whole pipeline inside each fold, so vocabularies and scalers never see the held-out fold.

In [3]:
def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))   # original loss units

def cross_validate(make_pipeline, frame=train, target_transform=None):
    """5 recorded folds; pipeline refit per fold. Returns fold MAEs, mean, sd, seconds."""
    start = time.perf_counter()
    fold_maes = []
    for fold in range(5):
        fit_rows, hold_rows = frame[frame["cv_fold"] != fold], frame[frame["cv_fold"] == fold]
        model = make_pipeline()
        y = fit_rows["loss"] if target_transform is None else target_transform[0](fit_rows["loss"])
        model.fit(fit_rows[FEATURES], y)
        pred = model.predict(hold_rows[FEATURES])
        if target_transform is not None:
            pred = target_transform[1](pred)
        fold_maes.append(mae(hold_rows["loss"], pred))
    return {"fold_maes": fold_maes, "cv_mae_mean": float(np.mean(fold_maes)),
            "cv_mae_sd": float(np.std(fold_maes, ddof=1)), "cv_seconds": time.perf_counter() - start}

def tail_report(model_name, y_true, y_pred, p90, p99):
    rows = []
    for label, mask in [("loss < train p90", y_true < p90), ("train p90 <= loss < p99", (y_true >= p90) & (y_true < p99)),
                        ("loss >= train p99", y_true >= p99), ("all validation claims", y_true == y_true)]:
        rows.append({"model": model_name, "partition": "validation", "segment": label,
                     "support_claims": int(mask.sum()), "mae_original_loss_units": mae(y_true[mask], y_pred[mask])})
    return rows

P90, P99 = train["loss"].quantile(0.90), train["loss"].quantile(0.99)
print(f"train p90 = {P90:.2f}, p99 = {P99:.2f}")

train p90 = 6401.74, p99 = 14024.14


## 3. Required baselines
Constants are fitted on train only and scored on validation with the same MAE function. These must reproduce the Gate 2 receipt (`1,962.92` and `1,805.09`).

In [4]:
baseline_rows = []
for name, fit_constant in [("training_mean_constant", np.mean), ("training_median_constant", np.median)]:
    constant = float(fit_constant(train["loss"]))
    fold_maes = []
    for fold in range(5):
        fit_rows, hold_rows = train[train["cv_fold"] != fold], train[train["cv_fold"] == fold]
        fold_maes.append(mae(hold_rows["loss"], np.full(len(hold_rows), fit_constant(fit_rows["loss"]))))
    baseline_rows.append({"model": name, "constant": constant, "cv_mae_mean": float(np.mean(fold_maes)),
                          "cv_mae_sd": float(np.std(fold_maes, ddof=1)),
                          "validation_mae": mae(validation["loss"], np.full(len(validation), constant))})
baselines = pd.DataFrame(baseline_rows)
MEAN_MAE, MEDIAN_MAE = baselines["validation_mae"].tolist()
assert abs(MEAN_MAE - 1962.918654) < 1e-3 and abs(MEDIAN_MAE - 1805.086353) < 1e-3
baselines

,model,constant,cv_mae_mean,cv_mae_sd,validation_mae
0,training_mean_constant,3038.840255,1968.122149,5.755773,1962.918654
1,training_median_constant,2115.390000,1810.477219,6.197761,1805.086353


## 4. Approved approaches (from `experiment_plan.md`)

| ID | Approach | Encoding | Prediction |
|---|---|---|---|
| E01 | Ridge regression | one-hot of pooled train vocabulary (`min_frequency=132`, unseen levels go to the rare bucket), standardized continuous fields | linear combination of inputs, in `loss` units |
| E02 | `HistGradientBoostingRegressor`, `loss="absolute_error"` | ordinal codes of the same pooled vocabulary with an explicit categorical mask over all 116 `cat` columns, unseen -> missing, no scaling | sum of many small trees, in `loss` units |

Hyperparameters are chosen by 5-fold CV inside train only. Validation is consulted once per final configuration.

In [5]:
def make_ridge(alpha):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=RARE_MIN_FREQUENCY, sparse_output=False), CAT),
        ("cont", StandardScaler(), CONT)])
    return Pipeline([("prep", prep), ("model", Ridge(alpha=alpha))])

CAT_MASK = np.array([True] * len(CAT) + [False] * len(CONT))   # must cover all 116 categorical columns
assert CAT_MASK.sum() == 116

def make_hgb(learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50, max_iter=200, loss="absolute_error"):
    prep = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               min_frequency=RARE_MIN_FREQUENCY), CAT),
        ("cont", "passthrough", CONT)])
    model = HistGradientBoostingRegressor(loss=loss, learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes,
                                          min_samples_leaf=min_samples_leaf, max_iter=max_iter,
                                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
                                          categorical_features=CAT_MASK, random_state=SEED)
    return Pipeline([("prep", prep), ("model", model)])

# pooling check: HGB accepts at most 255 levels per categorical field
probe = make_hgb().named_steps["prep"].fit(train[FEATURES])
enc = probe.named_transformers_["cat"]
pooled_sizes = [len(c) - (0 if inf is None else len(inf) - 1) for c, inf in zip(enc.categories_, enc.infrequent_categories_)]
print("max train levels per field before pooling:", max(len(c) for c in enc.categories_))
print("max levels per field after pooling:", max(pooled_sizes))
assert max(pooled_sizes) <= 255, "HGB accepts at most 255 levels per categorical field"
vocab_df = pd.DataFrame({"field": CAT, "train_levels": [len(c) for c in enc.categories_], "pooled_levels": pooled_sizes,
                         "levels_pooled_as_rare": [0 if inf is None else len(inf) for inf in enc.infrequent_categories_]})

max train levels per field before pooling: 306
max levels per field after pooling: 105


### 4a. Tune E01 (Ridge penalty) with the 5 training folds

In [6]:
ridge_grid = []
for alpha in [1, 10, 100, 1000]:
    res = cross_validate(lambda a=alpha: make_ridge(a))
    ridge_grid.append({"experiment": "E01", "alpha": alpha, **{k: v for k, v in res.items() if k != "fold_maes"}, "fold_maes": res["fold_maes"]})
    print(alpha, round(res["cv_mae_mean"], 2), round(res["cv_mae_sd"], 2), round(res["cv_seconds"], 1), "s")
ridge_grid = pd.DataFrame(ridge_grid)
BEST_ALPHA = int(ridge_grid.sort_values("cv_mae_mean").iloc[0]["alpha"])
print("selected alpha:", BEST_ALPHA, "| on grid edge:", BEST_ALPHA in (1, 1000))
ridge_grid

1 1312.84 7.22 112.0 s


10 1311.51 7.57 95.4 s


100 1308.55 7.82 145.8 s


1000 1309.02 8.24 103.6 s
selected alpha: 100 | on grid edge: False


,experiment,alpha,cv_mae_mean,cv_mae_sd,cv_seconds,fold_maes
0,E01,1,1312.836985,7.215074,111.957717,"[1314.0147268269757, 1313.503735514424, 1303.9..."
1,E01,10,1311.505803,7.570128,95.370914,"[1312.7575204321415, 1312.8219205691662, 1302...."
2,E01,100,1308.550909,7.822408,145.840204,"[1310.5953564933914, 1310.7656297623157, 1298...."
3,E01,1000,1309.019928,8.239244,103.583895,"[1312.7665338373022, 1312.2045229235534, 1299...."


### 4b. Tune E02 (boosting settings) with the 5 training folds
Small grid over learning rate, leaf count and minimum leaf size, as the plan specifies. Early stopping holds out 10% of the *fitting* rows only.

In [7]:
hgb_configs = [
    dict(learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50),
    dict(learning_rate=0.1, max_leaf_nodes=63, min_samples_leaf=50),
    dict(learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=200),
    dict(learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=50),
]
hgb_grid = []
for cfg in hgb_configs:
    res = cross_validate(lambda c=cfg: make_hgb(**c))
    hgb_grid.append({"experiment": "E02", **cfg, **{k: v for k, v in res.items() if k != "fold_maes"}, "fold_maes": res["fold_maes"]})
    print(cfg, round(res["cv_mae_mean"], 2), round(res["cv_mae_sd"], 2), round(res["cv_seconds"], 1), "s")
hgb_grid = pd.DataFrame(hgb_grid)
BEST_HGB = {k: hgb_grid.sort_values("cv_mae_mean").iloc[0][k] for k in ["learning_rate", "max_leaf_nodes", "min_samples_leaf"]}
BEST_HGB = {"learning_rate": float(BEST_HGB["learning_rate"]), "max_leaf_nodes": int(BEST_HGB["max_leaf_nodes"]), "min_samples_leaf": int(BEST_HGB["min_samples_leaf"])}
print("selected from CV:", BEST_HGB, "| max_leaf_nodes on grid edge:", BEST_HGB["max_leaf_nodes"] == 63)
hgb_grid

{'learning_rate': 0.1, 'max_leaf_nodes': 31, 'min_samples_leaf': 50} 1172.37 7.89 181.6 s


{'learning_rate': 0.1, 'max_leaf_nodes': 63, 'min_samples_leaf': 50} 1171.25 7.06 262.1 s


{'learning_rate': 0.1, 'max_leaf_nodes': 31, 'min_samples_leaf': 200} 1173.65 6.87 178.5 s


{'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 50} 1177.6 6.49 179.6 s
selected from CV: {'learning_rate': 0.1, 'max_leaf_nodes': 63, 'min_samples_leaf': 50} | max_leaf_nodes on grid edge: True


,experiment,learning_rate,max_leaf_nodes,min_samples_leaf,cv_mae_mean,cv_mae_sd,cv_seconds,fold_maes
0,E02,0.10,31,50,1172.367618,7.894725,181.581841,"[1178.8412724859102, 1169.0868061061406, 1168...."
1,E02,0.10,63,50,1171.249746,7.063284,262.120151,"[1176.408448265015, 1167.7185179338298, 1167.3..."
2,E02,0.10,31,200,1173.648828,6.867637,178.515248,"[1179.934040447367, 1172.7196681685268, 1168.9..."
3,E02,0.05,31,50,1177.598918,6.491075,179.567329,"[1183.9147712476697, 1176.6296321124642, 1171...."


## 5. First comparison: refit selected settings on all of train, score validation once
Each candidate is refit on the full training partition and scored on validation. Validation is consulted once per model here.

In [8]:
def fit_and_score(run_id, label, approach, settings, make_pipeline, cv_row, complexity, target_transform=None):
    start = time.perf_counter()
    model = make_pipeline()
    y = train["loss"] if target_transform is None else target_transform[0](train["loss"])
    model.fit(train[FEATURES], y)
    fit_seconds = time.perf_counter() - start
    pred = model.predict(validation[FEATURES])
    if target_transform is not None:
        pred = target_transform[1](pred)
    pred = np.asarray(pred, dtype=float)
    assert np.isfinite(pred).all(), "failed predictions"
    row = {"run_id": run_id, "model": label, "approach": approach, "settings": json.dumps(settings),
           "train_cv_mae": cv_row["cv_mae_mean"], "cv_mae_sd": cv_row["cv_mae_sd"],
           "validation_mae": mae(validation["loss"], pred), "validation_claims": len(validation),
           "complexity": complexity, "cv_seconds": cv_row["cv_seconds"], "final_fit_seconds": fit_seconds,
           "runtime_seconds": cv_row["cv_seconds"] + fit_seconds, "validation_consults": 1}
    row["pct_better_than_mean"] = 100 * (MEAN_MAE - row["validation_mae"]) / MEAN_MAE
    row["pct_better_than_median"] = 100 * (MEDIAN_MAE - row["validation_mae"]) / MEDIAN_MAE
    row["validation_minus_cv_mae"] = row["validation_mae"] - row["train_cv_mae"]
    return row, pred, model

best_ridge_cv = ridge_grid[ridge_grid.alpha == BEST_ALPHA].iloc[0]
ridge_row, ridge_pred, ridge_model = fit_and_score(
    "E01-ridge", "ridge_onehot_pooled", "regularized linear", {"alpha": BEST_ALPHA, "rare_min_frequency": RARE_MIN_FREQUENCY},
    lambda: make_ridge(BEST_ALPHA), best_ridge_cv,
    "")
n_ridge_columns = ridge_model.named_steps["prep"].transform(train[FEATURES].head(5)).shape[1]
ridge_row["complexity"] = f"{n_ridge_columns} input columns + intercept"

best_hgb_cv = hgb_grid.sort_values("cv_mae_mean").iloc[0]
hgb_row, hgb_pred, hgb_model = fit_and_score(
    "E02-hgb", "hist_gradient_boosting", "gradient-boosted trees", {**BEST_HGB, "loss": "absolute_error", "max_iter": 200, "early_stopping": True},
    lambda: make_hgb(**BEST_HGB), best_hgb_cv, "")
hgb_row["complexity"] = f"{hgb_model.named_steps['model'].n_iter_} boosting rounds, up to {BEST_HGB['max_leaf_nodes']} leaves each"
print(ridge_row["model"], round(ridge_row["validation_mae"], 2), "|", hgb_row["model"], round(hgb_row["validation_mae"], 2))

ridge_onehot_pooled 1290.43 | hist_gradient_boosting 1149.93


In [9]:
base_rows = []
for _, b in baselines.iterrows():
    r = {"run_id": "B01" if "mean" in b.model else "B02", "model": b.model, "approach": "constant", "settings": json.dumps({"constant": b.constant}),
         "train_cv_mae": b.cv_mae_mean, "cv_mae_sd": b.cv_mae_sd, "validation_mae": b.validation_mae, "validation_claims": len(validation),
         "complexity": "one fitted constant", "cv_seconds": 0.0, "final_fit_seconds": 0.0, "runtime_seconds": 0.0, "validation_consults": 1}
    r["pct_better_than_mean"] = 100 * (MEAN_MAE - r["validation_mae"]) / MEAN_MAE
    r["pct_better_than_median"] = 100 * (MEDIAN_MAE - r["validation_mae"]) / MEDIAN_MAE
    r["validation_minus_cv_mae"] = r["validation_mae"] - r["train_cv_mae"]
    base_rows.append(r)
first_comparison = pd.DataFrame(base_rows + [ridge_row, hgb_row])
first_comparison.round(2)

,run_id,model,approach,settings,train_cv_mae,cv_mae_sd,validation_mae,validation_claims,complexity,cv_seconds,final_fit_seconds,runtime_seconds,validation_consults,pct_better_than_mean,pct_better_than_median,validation_minus_cv_mae
0,B01,training_mean_constant,constant,"{""constant"": 3038.8402548133095}",1968.12,5.76,1962.92,28248,one fitted constant,0.00,0.00,0.00,1,0.00,-8.74,-5.20
1,B02,training_median_constant,constant,"{""constant"": 2115.39}",1810.48,6.20,1805.09,28248,one fitted constant,0.00,0.00,0.00,1,8.04,0.00,-5.39
2,E01-ridge,ridge_onehot_pooled,regularized linear,"{""alpha"": 100, ""rare_min_frequency"": 132}",1308.55,7.82,1290.43,28248,670 input columns + intercept,145.84,23.51,169.35,1,34.26,28.51,-18.12
3,E02-hgb,hist_gradient_boosting,gradient-boosted trees,"{""learning_rate"": 0.1, ""max_leaf_nodes"": 63, ""...",1171.25,7.06,1149.93,28248,"200 boosting rounds, up to 63 leaves each",262.12,53.24,315.36,1,41.42,36.29,-21.32


## 6. Error diagnostics across the target distribution (not a selection metric)
Segments use the **training** p90/p99 thresholds. Support counts are shown because the p99 segment has few claims.

In [10]:
tails = pd.DataFrame(
    tail_report("training_median_constant", validation["loss"].to_numpy(), np.full(len(validation), train["loss"].median()), P90, P99)
    + tail_report("training_mean_constant", validation["loss"].to_numpy(), np.full(len(validation), train["loss"].mean()), P90, P99)
    + tail_report(ridge_row["model"], validation["loss"].to_numpy(), ridge_pred, P90, P99)
    + tail_report(hgb_row["model"], validation["loss"].to_numpy(), hgb_pred, P90, P99))
tails.pivot(index="model", columns="segment", values="mae_original_loss_units").round(1), tails[tails.model == hgb_row["model"]][["segment", "support_claims"]]

(segment                   all validation claims  loss < train p90  \
 model                                                               
 hist_gradient_boosting                   1149.9             873.5   
 ridge_onehot_pooled                      1290.4            1050.7   
 training_mean_constant                   1962.9            1446.5   
 training_median_constant                 1805.1            1168.5   
 
 segment                   loss >= train p99  train p90 <= loss < p99  
 model                                                                 
 hist_gradient_boosting               9377.3                   3008.2  
 ridge_onehot_pooled                  9629.5                   2770.1  
 training_mean_constant              15170.4                   5670.3  
 training_median_constant            16093.8                   6593.7  ,
                     segment  support_claims
 12         loss < train p90           25422
 13  train p90 <= loss < p99            2547
 14       

A note on "validation consults": a consult is one newly fitted configuration scored on validation. Diagnostics below re-slice the same predictions and are not new consults. Baselines and the two first-round models are one consult each; the iteration adds one more consult for the iterated model.

## 7. Evidence-driven iteration (one controlled change)
**Evidence from the first comparison:** the selected boosted model used all 200 allowed boosting rounds, so early stopping never triggered and the model may be under-trained.

**Hypothesis:** raising `max_iter` from 200 to 400, with early stopping left on, lowers CV MAE.

**Changed variable:** `max_iter` only. Split, folds, seed, features, preprocessing, `learning_rate`, `max_leaf_nodes`, `min_samples_leaf`, and the MAE function stay fixed. The cell asserts this.

In [11]:
n_rounds_before = hgb_model.named_steps["model"].n_iter_
print("boosting rounds used before iteration:", n_rounds_before, "(cap was 200)")

ITER_CONFIG = {**BEST_HGB, "max_iter": 400}
before_cfg = json.loads(hgb_row["settings"])
changed = {k: (before_cfg[k], v) for k, v in {**ITER_CONFIG, "loss": "absolute_error", "early_stopping": True}.items() if before_cfg[k] != v}
assert list(changed) == ["max_iter"], f"iteration changed more than one variable: {changed}"

iter_cv = cross_validate(lambda: make_hgb(**ITER_CONFIG))
iter_row, iter_pred, iter_model = fit_and_score(
    "E02-hgb-iter1", "hist_gradient_boosting_max_iter_400", "gradient-boosted trees (iteration 1)",
    {**ITER_CONFIG, "loss": "absolute_error", "early_stopping": True}, lambda: make_hgb(**ITER_CONFIG), iter_cv, "")
n_rounds_after = iter_model.named_steps["model"].n_iter_
iter_row["complexity"] = f"{n_rounds_after} boosting rounds, up to {BEST_HGB['max_leaf_nodes']} leaves each"

cv_change = iter_row["train_cv_mae"] - hgb_row["train_cv_mae"]
val_change = iter_row["validation_mae"] - hgb_row["validation_mae"]
val_gain_pct = 100 * (hgb_row["validation_mae"] - iter_row["validation_mae"]) / hgb_row["validation_mae"]
stop = val_gain_pct < 1.0          # stopping rule from OCT-SPLIT-001: stop if gain over best previous < 1%
within_noise = abs(cv_change) < hgb_row["cv_mae_sd"]
iteration_table = pd.DataFrame({
    "changed_variable": ["max_iter"], "before": [200], "after": [400],
    "rounds_used_before": [n_rounds_before], "rounds_used_after": [n_rounds_after],
    "cv_mae_before": [hgb_row["train_cv_mae"]], "cv_mae_after": [iter_row["train_cv_mae"]],
    "validation_mae_before": [hgb_row["validation_mae"]], "validation_mae_after": [iter_row["validation_mae"]],
    "validation_change": [val_change], "validation_gain_pct": [val_gain_pct],
    "change_within_one_cv_sd": [within_noise], "stop_rule_met": [stop]})
iteration_table.round(3).T

boosting rounds used before iteration: 200 (cap was 200)


,0
changed_variable,max_iter
before,200
after,400
rounds_used_before,200
rounds_used_after,228
cv_mae_before,1171.25
cv_mae_after,1170.885
validation_mae_before,1149.934
validation_mae_after,1149.97
validation_change,0.035


## 8. Final comparison table
One table, all partitions labeled, original `loss` units. `train_cv_mae` is the mean over the 5 recorded training folds; `validation_mae` is the single validation score.

In [12]:
comparison = pd.concat([first_comparison, pd.DataFrame([iter_row])], ignore_index=True)
tails = pd.concat([tails, pd.DataFrame(tail_report(iter_row["model"], validation["loss"].to_numpy(), iter_pred, P90, P99))], ignore_index=True)
cols = ["run_id", "model", "train_cv_mae", "cv_mae_sd", "validation_mae", "validation_claims", "pct_better_than_mean",
        "pct_better_than_median", "complexity", "runtime_seconds", "validation_minus_cv_mae"]
comparison[cols].round(2)

,run_id,model,train_cv_mae,cv_mae_sd,validation_mae,validation_claims,pct_better_than_mean,pct_better_than_median,complexity,runtime_seconds,validation_minus_cv_mae
0,B01,training_mean_constant,1968.12,5.76,1962.92,28248,0.00,-8.74,one fitted constant,0.00,-5.20
1,B02,training_median_constant,1810.48,6.20,1805.09,28248,8.04,0.00,one fitted constant,0.00,-5.39
2,E01-ridge,ridge_onehot_pooled,1308.55,7.82,1290.43,28248,34.26,28.51,670 input columns + intercept,169.35,-18.12
3,E02-hgb,hist_gradient_boosting,1171.25,7.06,1149.93,28248,41.42,36.29,"200 boosting rounds, up to 63 leaves each",315.36,-21.32
4,E02-hgb-iter1,hist_gradient_boosting_max_iter_400,1170.89,7.16,1149.97,28248,41.42,36.29,"228 boosting rounds, up to 63 leaves each",339.38,-20.92


## 9. Checks: overfitting, stability, failed predictions, category handling

In [13]:
learned = comparison[comparison.approach != "constant"]
by_val, by_cv = learned.sort_values("validation_mae").iloc[0], learned.sort_values("train_cv_mae").iloc[0]
print("best learned model by validation MAE:", by_val["run_id"], "| by CV MAE:", by_cv["run_id"], "| agree:", by_val["run_id"] == by_cv["run_id"])

stability = comparison[["run_id", "train_cv_mae", "cv_mae_sd", "validation_mae", "validation_minus_cv_mae"]].copy()
stability["validation_gap_in_cv_sds"] = stability["validation_minus_cv_mae"] / stability["cv_mae_sd"]
print(stability.round(2).to_string(index=False))

preds = {"ridge": ridge_pred, "hgb": hgb_pred, "hgb_iter1": iter_pred}
failed = {k: {"non_finite": int((~np.isfinite(v)).sum()), "negative_predictions": int((v < 0).sum())} for k, v in preds.items()}
print("failed or negative predictions:", failed)

unseen = {c: int((~validation[c].isin(set(train[c]))).sum()) for c in CAT}
unseen = {k: v for k, v in unseen.items() if v}
print("validation claims with a level never seen in train, by field:", unseen)
print("Ridge routes unseen levels to the rare bucket (or all-zero one-hot); HGB treats them as missing.")

gap = ridge_row["validation_mae"] - hgb_row["validation_mae"]
print(f"HGB vs Ridge validation gap = {gap:.1f} = {gap / max(ridge_row['cv_mae_sd'], hgb_row['cv_mae_sd']):.1f} CV sds")
print(f"Iteration change = {val_change:+.2f} validation, {cv_change:+.2f} CV vs fold sd {hgb_row['cv_mae_sd']:.2f}; within noise: {within_noise}")

best learned model by validation MAE: E02-hgb | by CV MAE: E02-hgb-iter1 | agree: False
       run_id  train_cv_mae  cv_mae_sd  validation_mae  validation_minus_cv_mae  validation_gap_in_cv_sds
          B01       1968.12       5.76         1962.92                    -5.20                     -0.90
          B02       1810.48       6.20         1805.09                    -5.39                     -0.87
    E01-ridge       1308.55       7.82         1290.43                   -18.12                     -2.32
      E02-hgb       1171.25       7.06         1149.93                   -21.32                     -3.02
E02-hgb-iter1       1170.89       7.16         1149.97                   -20.92                     -2.92
failed or negative predictions: {'ridge': {'non_finite': 0, 'negative_predictions': 270}, 'hgb': {'non_finite': 0, 'negative_predictions': 0}, 'hgb_iter1': {'non_finite': 0, 'negative_predictions': 0}}


validation claims with a level never seen in train, by field: {'cat105': 1, 'cat110': 1, 'cat113': 1, 'cat116': 6}
Ridge routes unseen levels to the rare bucket (or all-zero one-hot); HGB treats them as missing.
HGB vs Ridge validation gap = 140.5 = 18.0 CV sds
Iteration change = +0.04 validation, -0.36 CV vs fold sd 7.06; within noise: True


In [14]:
print(tails.pivot_table(index="model", columns="segment", values="mae_original_loss_units").round(1).to_string())
print(tails[tails.model == iter_row["model"]][["segment", "support_claims"]].to_string(index=False))

segment                              all validation claims  loss < train p90  loss >= train p99  train p90 <= loss < p99
model                                                                                                                   
hist_gradient_boosting                              1149.9             873.5             9377.3                   3008.2
hist_gradient_boosting_max_iter_400                 1150.0             874.2             9350.7                   3004.3
ridge_onehot_pooled                                 1290.4            1050.7             9629.5                   2770.1
training_mean_constant                              1962.9            1446.5            15170.4                   5670.3
training_median_constant                            1805.1            1168.5            16093.8                   6593.7
                segment  support_claims
       loss < train p90           25422
train p90 <= loss < p99            2547
      loss >= train p99          

## 10. Cross-check against a teammate's independent run
Jonathan Deng's branch `jonathan` ran Ridge, HGB and a random forest on the same split with different preprocessing (no rare-level pooling, ordinal codes without a categorical mask). This compares results only; none of his code or numbers feed this notebook.

In [15]:
def read_branch_csv(ref_path):
    try:
        text = subprocess.run(["git", "show", ref_path], capture_output=True, text=True, check=True, cwd=ROOT).stdout
        return pd.read_csv(io.StringIO(text))
    except Exception as exc:
        print("teammate results not available locally:", exc)
        return None

jon = read_branch_csv("origin/jonathan:notebooks/jonathan-deng/october_modeling_outputs/model_comparison.csv")
xcheck = None
if jon is not None:
    jon = jon[["model", "training_or_cv_mae_original_units", "validation_mae_original_units"]].rename(
        columns={"training_or_cv_mae_original_units": "jd_cv_mae", "validation_mae_original_units": "jd_validation_mae"})
    mine = comparison[["model", "train_cv_mae", "validation_mae"]].rename(columns={"model": "my_model"})
    pairs = [("training_mean_constant", "training_mean_constant"), ("training_median_constant", "training_median_constant"),
             ("ridge_one_hot", "ridge_onehot_pooled"), ("hist_gradient_boosting_v1", "hist_gradient_boosting"),
             ("hist_gradient_boosting_v2", "hist_gradient_boosting_max_iter_400")]
    xcheck = pd.DataFrame([{**jon[jon.model == j].iloc[0].to_dict(), **mine[mine.my_model == m].iloc[0].to_dict()} for j, m in pairs])
    xcheck["validation_difference"] = xcheck["validation_mae"] - xcheck["jd_validation_mae"]
    print(xcheck.round(2).to_string(index=False))

                    model  jd_cv_mae  jd_validation_mae                            my_model  train_cv_mae  validation_mae  validation_difference
   training_mean_constant    1968.12            1962.92              training_mean_constant       1968.12         1962.92                   0.00
 training_median_constant    1810.48            1805.09            training_median_constant       1810.48         1805.09                   0.00
            ridge_one_hot    1307.37            1288.81                 ridge_onehot_pooled       1308.55         1290.43                   1.62
hist_gradient_boosting_v1    1203.84            1187.53              hist_gradient_boosting       1171.25         1149.93                 -37.60
hist_gradient_boosting_v2    1224.38            1209.22 hist_gradient_boosting_max_iter_400       1170.89         1149.97                 -59.25


## 11. Experiment register, comparison table and run metadata

In [16]:
git_commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip()
dirty_lines = [l for l in subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=ROOT).stdout.splitlines()
               if "junaid-pathan/october" not in l]
git_dirty = bool(dirty_lines)
lib = "; ".join(f"{k} {v}" for k, v in versions.items())

register = []
def reg(**kw):
    base = {"workflow_run": WORKFLOW_RUN, "split_version": "OCT-SPLIT-001", "seed": SEED, "code_version_commit": git_commit,
            "git_dirty": git_dirty, "library_versions": lib, "train_claims": len(train), "validation_claims": len(validation),
            "feature_set": "cat1-cat116 + cont1-cont14 (130 predictors; id and loss excluded)"}
    register.append({**base, **kw})

for _, b in comparison[comparison.approach == "constant"].iterrows():
    reg(experiment_id=b.run_id, iteration=0, status="success", changed_variable="none (required reference)",
        hypothesis="Required reference point." if "mean" in b.model else "Median minimizes absolute error for a constant, so it is the tougher bar.",
        model_config=b.settings, preprocessing="none", cv_mae_mean=b.train_cv_mae, cv_mae_sd=b.cv_mae_sd, validation_mae=b.validation_mae,
        pct_vs_median=b.pct_better_than_median, pct_vs_mean=b.pct_better_than_mean, runtime_seconds=0.0,
        result=f"Validation MAE {b.validation_mae:.2f}", interpretation="Reference only; no learned relationship.",
        next_decision="Retain as the bar every model must beat.")
for _, g in ridge_grid.iterrows():
    sel = int(g.alpha) == BEST_ALPHA
    reg(experiment_id=f"E01-alpha{int(g.alpha)}", iteration=1, status="selected" if sel else "not_selected",
        hypothesis="Ridge with one-hot pooled categoricals beats the median constant.", changed_variable=f"alpha={int(g.alpha)}",
        preprocessing=f"one-hot min_frequency={RARE_MIN_FREQUENCY}, StandardScaler on cont", model_config=json.dumps({"alpha": int(g.alpha)}),
        cv_mae_mean=g.cv_mae_mean, cv_mae_sd=g.cv_mae_sd, validation_mae=ridge_row["validation_mae"] if sel else None,
        pct_vs_median=ridge_row["pct_better_than_median"] if sel else None, pct_vs_mean=ridge_row["pct_better_than_mean"] if sel else None,
        runtime_seconds=g.cv_seconds, result=f"CV MAE {g.cv_mae_mean:.2f} +/- {g.cv_mae_sd:.2f}",
        interpretation="Lowest CV MAE in the grid; carried to validation." if sel else "Higher CV MAE than the selected alpha; not scored on validation.",
        next_decision="Compare against E02." if sel else "Dropped from further runs.")
for _, g in hgb_grid.iterrows():
    cfg = {k: (float(g[k]) if k == "learning_rate" else int(g[k])) for k in ["learning_rate", "max_leaf_nodes", "min_samples_leaf"]}
    sel = cfg == BEST_HGB
    reg(experiment_id="E02-lr{learning_rate}-leaf{max_leaf_nodes}-min{min_samples_leaf}".format(**cfg), iteration=1,
        status="selected" if sel else "not_selected",
        hypothesis="Boosted trees with absolute-error loss beat Ridge.", changed_variable=", ".join(f"{k}={v}" for k, v in cfg.items()),
        preprocessing=f"ordinal codes min_frequency={RARE_MIN_FREQUENCY}, categorical mask over 116 cat columns, unseen -> missing",
        model_config=json.dumps({**cfg, "loss": "absolute_error", "max_iter": 200}), cv_mae_mean=g.cv_mae_mean, cv_mae_sd=g.cv_mae_sd,
        validation_mae=hgb_row["validation_mae"] if sel else None, pct_vs_median=hgb_row["pct_better_than_median"] if sel else None,
        pct_vs_mean=hgb_row["pct_better_than_mean"] if sel else None, runtime_seconds=g.cv_seconds,
        result=f"CV MAE {g.cv_mae_mean:.2f} +/- {g.cv_mae_sd:.2f}",
        interpretation="Lowest CV MAE in the grid; carried to validation." if sel else "Higher CV MAE than the selected setting; not scored on validation.",
        next_decision="Evidence-driven iteration on max_iter." if sel else "Dropped from further runs.")
reg(experiment_id=iter_row["run_id"], iteration=2, status="success",
    hypothesis="The selected HGB used all 200 rounds (early stopping never fired), so raising max_iter lowers CV MAE.",
    changed_variable="max_iter: 200 -> 400 (only change)", preprocessing="unchanged", model_config=iter_row["settings"],
    cv_mae_mean=iter_row["train_cv_mae"], cv_mae_sd=iter_row["cv_mae_sd"], validation_mae=iter_row["validation_mae"],
    pct_vs_median=iter_row["pct_better_than_median"], pct_vs_mean=iter_row["pct_better_than_mean"], runtime_seconds=iter_row["runtime_seconds"],
    result=f"CV {hgb_row['train_cv_mae']:.2f} -> {iter_row['train_cv_mae']:.2f}; validation {hgb_row['validation_mae']:.2f} -> {iter_row['validation_mae']:.2f} ({val_gain_pct:+.2f}%)",
    interpretation=("Change is smaller than one CV fold SD; treat as no reliable difference." if within_noise else "Change exceeds one CV fold SD.")
        + f" Rounds used {n_rounds_before} -> {n_rounds_after}.",
    next_decision="Stop: gain below the 1% stopping rule." if stop else "Gain meets the 1% rule; one more justified iteration is allowed.")
register = pd.DataFrame(register)

comparison.assign(train_cv_mae_partition="train 5-fold CV, original loss units",
                  validation_mae_partition="validation, original loss units").to_csv(OUT / "model_comparison.csv", index=False)
register.to_csv(OUT / "experiment_register.csv", index=False)
pd.concat([ridge_grid.assign(grid="E01"), hgb_grid.assign(grid="E02")]).to_csv(OUT / "tuning_grid_results.csv", index=False)
tails.to_csv(OUT / "tail_error_diagnostics.csv", index=False)
vocab_df.to_csv(OUT / "pooled_vocabulary.csv", index=False)
pd.DataFrame({"id": validation["id"], "partition": "validation", "loss": validation["loss"],
              "pred_training_mean": train["loss"].mean(), "pred_training_median": train["loss"].median(),
              "pred_ridge": ridge_pred, "pred_hgb": hgb_pred, "pred_hgb_iter1": iter_pred}).to_csv(OUT / "validation_predictions.csv", index=False)
if xcheck is not None:
    xcheck.to_csv(OUT / "teammate_crosscheck.csv", index=False)
metadata = {"workflow_run": WORKFLOW_RUN, "modeling_data_version": "OCT-MD-001", "split_version": "OCT-SPLIT-001", "seed": SEED,
            "source_sha256": EXPECTED_SOURCE_SHA, "manifest_sha256": EXPECTED_MANIFEST_SHA,
            "partition_counts_used": {"train": len(train), "validation": len(validation), "test": "not loaded for modeling"},
            "cv_folds": 5, "rare_min_frequency": RARE_MIN_FREQUENCY, "metric": "MAE, original loss units",
            "train_p90": float(P90), "train_p99": float(P99), "git_commit": git_commit, "git_dirty": git_dirty,
            "library_versions": versions, "total_runtime_seconds_models": float(comparison["runtime_seconds"].sum()),
            "validation_consults": {"B01": 1, "B02": 1, "E01-ridge": 1, "E02-hgb": 1, "E02-hgb-iter1": 1},
            "test_policy": "test rows dropped at load; no target summary, prediction or score"}
(OUT / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
print(sorted(p.name for p in OUT.iterdir()))
register[["experiment_id", "status", "changed_variable", "cv_mae_mean", "validation_mae", "next_decision"]]

['experiment_register.csv', 'model_comparison.csv', 'pooled_vocabulary.csv', 'run_metadata.json', 'tail_error_diagnostics.csv', 'teammate_crosscheck.csv', 'tuning_grid_results.csv', 'validation_predictions.csv']


,experiment_id,status,changed_variable,cv_mae_mean,validation_mae,next_decision
0,B01,success,none (required reference),1968.122149,1962.918654,Retain as the bar every model must beat.
1,B02,success,none (required reference),1810.477219,1805.086353,Retain as the bar every model must beat.
2,E01-alpha1,not_selected,alpha=1,1312.836985,NaN,Dropped from further runs.
3,E01-alpha10,not_selected,alpha=10,1311.505803,NaN,Dropped from further runs.
4,E01-alpha100,selected,alpha=100,1308.550909,1290.431830,Compare against E02.
5,E01-alpha1000,not_selected,alpha=1000,1309.019928,NaN,Dropped from further runs.
6,E02-lr0.1-leaf31-min50,not_selected,"learning_rate=0.1, max_leaf_nodes=31, min_samp...",1172.367618,NaN,Dropped from further runs.
7,E02-lr0.1-leaf63-min50,selected,"learning_rate=0.1, max_leaf_nodes=63, min_samp...",1171.249746,1149.934472,Evidence-driven iteration on max_iter.
8,E02-lr0.1-leaf31-min200,not_selected,"learning_rate=0.1, max_leaf_nodes=31, min_samp...",1173.648828,NaN,Dropped from further runs.
9,E02-lr0.05-leaf31-min50,not_selected,"learning_rate=0.05, max_leaf_nodes=31, min_sam...",1177.598918,NaN,Dropped from further runs.


## 12. Model explanations for the Challenge Advisor

**Inputs (both models):** 116 anonymous categorical fields and 14 continuous fields. `id` and `loss` are never inputs. Categorical levels seen on fewer than 132 training claims are pooled into one "rare" level; the pooling is learned from training claims only.

| | Ridge (E01) | Gradient-boosted trees (E02) |
|---|---|---|
| Encoding | One 0/1 column per pooled level, about 670 inputs; continuous fields standardized | One integer code per pooled level, marked as unordered categories; continuous fields as-is |
| Key setting | `alpha` (penalty strength), chosen by training CV | learning rate, leaves per tree, minimum claims per leaf, number of rounds |
| Output | A weighted sum of the inputs: a predicted `loss` in the original units | Sum of many small decision trees: a predicted `loss` in the original units |
| How it learns | Finds one weight per input that best fits training losses, shrinking weights to avoid overfitting | Adds trees one at a time, each correcting the previous trees' errors, minimizing absolute error |
| Unseen levels | Counted as the rare level | Treated as missing |

**Why the tail matters:** MAE rewards getting the many ordinary claims right. The large-claim segments contain far fewer claims (see the support counts), so their error estimates are noisy. They are diagnostics, not the selection metric.

**Limits:** September EDA used the full labeled file, so validation is not wholly unseen; one split cannot show stability over time; the p99 segment has only 279 validation claims; the test partition was not used.